# Maia Corpus v9 — Code Acquisition Engine v1.0
Production acquisition, Hugging Face first. Original code, exact SHA-256 deduplication, provisional license screening, persistent immutable batches. No code is executed. No automatic legal guarantee. Default acquisition is Python via CodeParrot; optional authorized Stack v1 dedup adds other languages. No automatic access requests. Global deduplication and final tokenizer are later stages.
Sources: https://huggingface.co/datasets/codeparrot/codeparrot-clean and https://huggingface.co/datasets/bigcode/the-stack-dedup .

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, os, time, re, sqlite3, gzip, collections, shutil
from huggingface_hub import HfApi, hf_hub_download
from datasets import load_dataset
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_4_0/maia-corpus-v9')/'code_v10'
ROOT.mkdir(parents=True,exist_ok=True)
# Proxy only: code tends to tokenize differently from prose.
CHARS_PER_TOKEN=3.0
TARGETS={'Python':360_000_000,'JavaScript':240_000_000,'C':180_000_000,'C++':180_000_000,'Java':120_000_000,'TypeScript':120_000_000}
ALLOW={'mit','apache-2.0','bsd-2-clause','bsd-3-clause','isc','0bsd','zlib','unlicense','cc0-1.0'}
ENABLE_STACK=False # Enable only after obtaining access and accepting upstream terms.
BATCH=2000
MAX_CHARS=500_000
SEED=2026
api=HfApi()
try:
    from google.colab import userdata
    TOKEN=userdata.get('HF_TOKEN')
except Exception: TOKEN=None
print('Target:',sum(TARGETS.values()),'proxy tokens; root:',ROOT)


## 1. Resolve sources and preserve dataset cards. The Stack is optional and may be gated. The default public source covers Python only; missing language quotas remain explicit. No silent substitution.

In [ ]:
def atomic(path,obj):
    tmp=path.with_suffix(path.suffix+'.part')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8'); os.replace(tmp,path)
def retry(fn):
    for i in range(3):
        try:return fn()
        except Exception as e:
            print('RETRY',i+1,type(e).__name__,str(e)[:200],flush=True)
            if i==2:raise
            time.sleep(5*(i+1))
planpath=ROOT/'plan.json'
config={'targets':TARGETS,'ratio':CHARS_PER_TOKEN,'allow':sorted(ALLOW),'stack':ENABLE_STACK,'batch':BATCH,'max_chars':MAX_CHARS,'seed':SEED}
if planpath.exists():
    plan=json.loads(planpath.read_text()); assert plan['config']==config,'Configuration changed: use new output directory or original configuration.'
else:
    plan={'config':config,'sources':[]}
    repos=['codeparrot/codeparrot-clean-train']+(['bigcode/the-stack-dedup'] if ENABLE_STACK else [])
    for repo in repos:
        try:
            info=retry(lambda:api.dataset_info(repo,token=TOKEN,files_metadata=True,timeout=30))
            card=retry(lambda:hf_hub_download(repo,'README.md',repo_type='dataset',revision=info.sha,token=TOKEN))
            tag=repo.replace('/','__'); shutil.copyfile(card,ROOT/(tag+'.README.md'))
            files=[x.rfilename for x in info.siblings if x.rfilename.endswith(('.json.gz','.jsonl.gz','.parquet','.jsonl'))]
            if repo.startswith('bigcode/'):
                files=[f for f in files if any('/'+s+'/' in '/'+f.lower()+'/' for s in ['python','javascript','c','cpp','java','typescript'])]
            assert files,'No matching data shards'
            import random
            random.Random(SEED).shuffle(files)
            plan['sources'].append({'repo':repo,'revision':info.sha,'files':files})
            print(repo,info.sha,len(files),flush=True)
        except Exception as e:print('SOURCE BLOCKED',repo,str(e)[:300],flush=True)
    assert plan['sources'],'No source accessible'
    atomic(planpath,plan)


## 2. Local index rebuilt from committed batches. Each batch stores its cursor and records together, so an interrupted write cannot advance the cursor. Resume replays only the unfinished source file (skipping committed rows).

In [ ]:
db=sqlite3.connect(':memory:')
db.execute('create table seen(hash text primary key)')
totals=collections.Counter(); rejects=collections.Counter(); cursors={}; batches=ROOT/'batches'; batches.mkdir(exist_ok=True)
for p in sorted(batches.glob('*.json.gz')):
    with gzip.open(p,'rt',encoding='utf-8') as f: obj=json.load(f)
    key=obj['key']; assert obj['start']==cursors.get(key,0),f'Noncontiguous checkpoint {p}'
    for r in obj['records']:
        assert hashlib.sha256(r['text'].encode()).hexdigest()==r['sha256'],p
        db.execute('insert into seen values (?)',(r['sha256'],)); totals[r['language']]+=r['proxy_tokens']
    rejects.update(obj['rejects']); cursors[key]=obj['end']
db.commit(); print('Recovered',dict(totals),'batches',len(list(batches.glob('*.json.gz'))))


In [ ]:
EXT={'.py':'Python','.js':'JavaScript','.jsx':'JavaScript','.ts':'TypeScript','.tsx':'TypeScript','.c':'C','.cpp':'C++','.cc':'C++','.cxx':'C++','.java':'Java'}
SECRET=re.compile(r'-----BEGIN (?:RSA |EC |OPENSSH |DSA )?PRIVATE KEY-----|\b(?:ghp_|github_pat_|sk_live_)[A-Za-z0-9_]{16,}|\bAKIA[A-Z0-9]{16}\b')
SPDX=re.compile(r'SPDX-License-Identifier:\s*([^\r\n*]+)',re.I)
def screen(row,src,key,idx):
    text=row.get('content'); path=row.get('path') or row.get('max_stars_repo_path'); repo=row.get('repo_name') or row.get('max_stars_repo_name')
    lic=row.get('license') or row.get('max_stars_repo_licenses')
    licenses=[lic] if isinstance(lic,str) else lic if isinstance(lic,list) else []
    if not licenses or any(not isinstance(x,str) or x.lower().strip() not in ALLOW for x in licenses):return None,'license_unknown_or_excluded'
    if not isinstance(path,str) or not isinstance(repo,str) or not repo:return None,'missing_provenance'
    lang=EXT.get(Path(path).suffix.lower())
    if lang not in TARGETS or totals[lang]>=TARGETS[lang]:return None,'language_or_quota'
    if not isinstance(text,str) or not 100<=len(text)<=MAX_CHARS or '\x00' in text:return None,'size_or_binary'
    if row.get('autogenerated') is True or any(len(x)>1500 for x in text.splitlines()):return None,'generated_or_long_line'
    if SECRET.search(text):return None,'possible_secret'
    if any(x.strip().lower() not in ALLOW for x in SPDX.findall(text)):return None,'conflicting_spdx'
    h=hashlib.sha256(text.encode()).hexdigest()
    if db.execute('select 1 from seen where hash=?',(h,)).fetchone():return None,'duplicate'
    return {'text':text,'sha256':h,'language':lang,'license_evidence':licenses,'rights_status':'PERMISSIVE_SOURCE_CANDIDATE','repository':repo,'path':path,'source':src['repo'],'revision':src['revision'],'source_file':key,'row':idx,'commit':row.get('max_stars_repo_head_hexsha'),'characters':len(text),'proxy_tokens':round(len(text)/CHARS_PER_TOKEN)},None
# Local synthetic tests; no network, no pilot acquisition.
r={'content':'print(1)\n'*30,'path':'a.py','repo_name':'example/test','license':'mit'}
a,why=screen(r,{'repo':'test','revision':'test'},'test',0)
if totals['Python']<TARGETS['Python']:assert a and not why
assert screen({**r,'license':'gpl-3.0'},{},'',0)[1]=='license_unknown_or_excluded'
print('Filter tests OK')


In [ ]:
def commit(key,start,end,records,counts):
    ident=hashlib.sha256(key.encode()).hexdigest()[:24]
    path=batches/f'{ident}_{start:012d}.json.gz'; tmp=path.with_suffix('.part')
    with gzip.open(tmp,'wt',encoding='utf-8',compresslevel=3) as f:json.dump({'key':key,'start':start,'end':end,'records':records,'rejects':dict(counts)},f,ensure_ascii=False)
    os.replace(tmp,path); cursors[key]=end; rejects.update(counts); db.commit()
    print('COMMIT',key,end,'accepted',len(records),'proxy',dict(totals),flush=True)
def complete():return all(totals[k]>=v for k,v in TARGETS.items())
finished_path=ROOT/'finished.json'; finished=set(json.loads(finished_path.read_text())) if finished_path.exists() else set()
errors=[]
for src in plan['sources']:
    for file in src['files']:
        if complete() or (src['repo'].startswith('codeparrot/') and totals['Python']>=TARGETS['Python']):break
        key=src['repo']+'@'+src['revision']+'/'+file
        if key in finished:continue
        try:
            local=retry(lambda:hf_hub_download(src['repo'],file,repo_type='dataset',revision=src['revision'],token=TOKEN,etag_timeout=30))
            kind='parquet' if file.endswith('.parquet') else 'json'
            ds=load_dataset(kind,data_files=local,split='train',streaming=True)
            start=cursors.get(key,0); records=[]; counts=collections.Counter(); end=start
            print('READ',file,'resume',start,flush=True)
            for idx,row in enumerate(ds):
                if idx<start:continue
                rec,why=screen(row,src,key,idx)
                if rec:
                    db.execute('insert into seen values (?)',(rec['sha256'],)); totals[rec['language']]+=rec['proxy_tokens']; records.append(rec)
                else:counts[why]+=1
                end=idx+1
                if end-start>=BATCH or complete():
                    commit(key,start,end,records,counts);start=end;records=[];counts=collections.Counter()
                if complete() or (src['repo'].startswith('codeparrot/') and totals['Python']>=TARGETS['Python']):break
            if end>start:commit(key,start,end,records,counts)
            finished.add(key);atomic(finished_path,sorted(finished))
            # Delete only local downloaded shard, not persistent corpus.
            # HF cache cleanup is left to the runtime; avoid deleting shared cache blobs.
        except Exception as e:
            # In-memory state may contain uncommitted records. Fail immediately; restart from cell 2.
            errors.append({'key':key,'error':str(e)[:500]});atomic(ROOT/'errors.json',errors)
            raise RuntimeError('Interrupted source; rerun index recovery then acquisition. Committed batches preserved.') from e


In [ ]:
report={'status':'TARGET_MET' if complete() else 'PARTIAL_TARGET','proxy_tokens':dict(totals),'target':TARGETS,'deficits':{k:max(0,v-totals[k]) for k,v in TARGETS.items()},'accepted_documents':db.execute('select count(*) from seen').fetchone()[0],'rejections':dict(rejects),'completed_files':len(finished),'rights_status':'PERMISSIVE_SOURCE_CANDIDATES_ONLY','requires_final_tokenizer':True}
atomic(ROOT/'summary.json',report);print(json.dumps(report,indent=2))


## Limitations
Repository license labels are not file-level legal proofs. Preserve original notices and satisfy attribution obligations before release. Secret screening is heuristic; no malware execution, semantic dedup, benchmark decontamination or global v8 comparison is performed. Default source is Python only. Stack v1 access, current terms and opt-outs must be validated before enabling; use a separate output directory if changing the configuration. Token counts are provisional. No live Colab run was performed.